# VDF inspection and dataset test for one timestep

Notebook is used to inspect velocity distribution functions (VDFs) from one Vlasiator timestep.

The idea is to inspect VDFs from two selected region:
- X-point sample taken close to the magnetotail reconnection region where a non-Maxwellian distribution is expected.
- Lobe sample taken from lobes where Maxwellian distribution is expected.

Notebook has two purposes:

1. **Visual inspection**
    Check if the choosen regions produce the expected VDF shapes.

2. **Data format testing**
    Experiment with data format used to save the extracted data and inspecting the format.

This notebook is exploratory

In [1]:
"""Inspect and save VDF samples from one Vlasiator timestep.

Notebook loads one Vlasiator `.vlsv` file, selects one VDF from an X point
and lobe region, plots them, and saves them as small dataset, which is then inspected.
"""
import os
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


os.environ['PTNOLATEX']='1'
sys.path.insert(0, "/home/jysa/vdf-ml/externals/analysator")

import analysator as pt

In [2]:
from pathlib import Path
import re
import analysator as pt

BASE = Path("/wrk-vakka/group/spacephysics/vlasiator")

ROOTS = {
    "2d": BASE / "2D",
    "3d": BASE / "3D",
}

# Only names like BCH, EGI, DCA, FIG, etc.
RUN_ID_RE = re.compile(r"^[A-Z]{3}$")

# Use "latest" to inspect one representative file per run.
# Use "all" to inspect every matching .vlsv file.
FILE_MODE = "latest"      # "latest" or "all"
MAX_FILES_PER_RUN = 1     # set to None if FILE_MODE = "all"


def frame_number(path):
    """
    Extract frame number from names like:
      bulk.0003475.vlsv
      bulk1.0001340.vlsv
      bulk1.egl.0001183.vlsv
    """
    match = re.search(r"\.(\d+)\.vlsv$", path.name)
    if match:
        return int(match.group(1))
    return -1


def is_three_letter_run_dir(path):
    return path.is_dir() and RUN_ID_RE.fullmatch(path.name) is not None


def find_vlsv_files(run_dir):
    """
    Find likely bulk .vlsv files inside one run directory.

    Handles paths like:
      2D/BCH/bulk/bulk.0003475.vlsv
      3D/EGI/bulk/dense_cold_hall1e5_afterRestart374/bulk1.0001340.vlsv
    """
    patterns = [
        "bulk/bulk*.vlsv",
        "bulk/**/*.vlsv",
        "**/bulk*.vlsv",
    ]

    found = []
    seen = set()

    for pattern in patterns:
        try:
            matches = run_dir.glob(pattern)
            for path in matches:
                key = str(path)
                if path.is_file() and key not in seen:
                    seen.add(key)
                    found.append(path)
        except OSError as exc:
            print(f"Could not scan {run_dir} with pattern {pattern}: {exc}")

    return sorted(found, key=lambda p: (frame_number(p), str(p)))


def choose_files(vlsv_files):
    if not vlsv_files:
        return []

    if FILE_MODE == "latest":
        return [vlsv_files[-1]]

    if FILE_MODE == "all":
        if MAX_FILES_PER_RUN is None:
            return vlsv_files
        return vlsv_files[:MAX_FILES_PER_RUN]

    raise ValueError('FILE_MODE must be either "latest" or "all"')


def inspect_file(dim_name, run_id, path):
    print("\n===", dim_name, run_id, "===")
    print("file:", path)

    try:
        reader = pt.vlsvfile.VlsvReader(str(path))
    except Exception as exc:
        print("VlsvReader error:", exc)
        return

    print("active_populations:", getattr(reader, "active_populations", None))

    for pop in ["avgs", "proton", "protons"]:
        try:
            print(pop, "check_population:", reader.check_population(pop))
        except Exception as exc:
            print(pop, "check_population error:", exc)

    for param in ["vxblocks_ini", "vxblocks", "vxmin", "vxmax"]:
        try:
            print(param, "check_parameter:", reader.check_parameter(param))
        except Exception as exc:
            print(param, "check_parameter error:", exc)

    for pop in ["avgs", "proton", "protons"]:
        try:
            print(pop, "velocity mesh size:", reader.get_velocity_mesh_size(pop))
        except Exception as exc:
            print(pop, "velocity mesh size error:", exc)

    try:
        variables = reader.get_variables()
        print("first variables:", variables[:50])
        print(
            "vdf/block-ish variables:",
            [
                v for v in variables
                if "block" in v.lower()
                or "vdf" in v.lower()
                or "vg_" in v.lower()
            ][:100],
        )
    except Exception as exc:
        print("get_variables error:", exc)


for dim_name, root in ROOTS.items():
    print("\n\n############################")
    print("Scanning", dim_name, "root:", root)
    print("############################")

    if not root.exists():
        print("Missing root:", root)
        continue

    try:
        run_dirs = sorted(
            [path for path in root.iterdir() if is_three_letter_run_dir(path)],
            key=lambda p: p.name,
        )
    except OSError as exc:
        print("Could not list root:", root, exc)
        continue

    print("three-letter runs found:", [p.name for p in run_dirs])

    for run_dir in run_dirs:
        vlsv_files = find_vlsv_files(run_dir)

        if not vlsv_files:
            print("\n===", dim_name, run_dir.name, "===")
            print("No matching .vlsv bulk files found under:", run_dir)
            continue

        for vlsv_file in choose_files(vlsv_files):
            inspect_file(dim_name, run_dir.name, vlsv_file)



############################
Scanning 2d root: /wrk-vakka/group/spacephysics/vlasiator/2D
############################
three-letter runs found: ['ABA', 'ABC', 'AEA', 'AEC', 'AFC', 'AGB', 'AGC', 'AGD', 'AGE', 'AGF', 'AIA', 'AIB', 'AIC', 'AID', 'BCG', 'BCH', 'BCQ', 'BCV', 'BED', 'BFA', 'BFB', 'BFD', 'BGA', 'BGC', 'BGD', 'BGE', 'BGF', 'BGG', 'BGH', 'BHA', 'BIA', 'BIB', 'BIC', 'BID', 'BIE', 'BIF', 'BJA']

=== 2d ABA ===
file: /wrk-vakka/group/spacephysics/vlasiator/2D/ABA/bulk/bulk.0001019.vlsv


INFO: Found population avgs
INFO: Found population proton
INFO: Found population protons


active_populations: ['avgs']
avgs check_population: True
proton check_population: False
protons check_population: False
vxblocks_ini check_parameter: True
vxblocks check_parameter: False
vxmin check_parameter: True
vxmax check_parameter: True
avgs velocity mesh size: [67 67 67]
proton velocity mesh size: [67 67 67]
protons velocity mesh size: [67 67 67]
first variables: ['B', 'BGB_vol', 'B_vol', 'Blocks', 'Boundary_layer', 'Boundary_type', 'CellID', 'DX', 'DY', 'DZ', 'E', 'EXHALL_000_100', 'EXHALL_001_101', 'EXHALL_010_110', 'EXHALL_011_111', 'EYHALL_000_010', 'EYHALL_001_011', 'EYHALL_100_110', 'EYHALL_101_111', 'EZHALL_000_001', 'EZHALL_010_011', 'EZHALL_100_101', 'EZHALL_110_111', 'E_vol', 'MPI_rank', 'PERB_vol', 'PTensorDiagonal', 'PTensorOffDiagonal', 'Pressure_from_solver', 'RhoBackstream', 'RhoNonBackstream', 'RhoVBackstream', 'RhoVNonBackstream', 'X', 'Y', 'Z', 'background_B', 'fSaved', 'perturbed_B', 'rho', 'rho_v']
vdf/block-ish variables: ['Blocks']


INFO: Found population avgs
INFO: Found population proton
INFO: Found population protons



=== 2d ABC ===
file: /wrk-vakka/group/spacephysics/vlasiator/2D/ABC/bulk/bulk.0001371.vlsv
active_populations: ['avgs']
avgs check_population: True
proton check_population: False
protons check_population: False
vxblocks_ini check_parameter: True
vxblocks check_parameter: False
vxmin check_parameter: True
vxmax check_parameter: True
avgs velocity mesh size: [67 67 67]
proton velocity mesh size: [67 67 67]
protons velocity mesh size: [67 67 67]
first variables: ['B', 'B_vol', 'Blocks', 'Boundary_layer', 'Boundary_type', 'CellID', 'DX', 'DY', 'DZ', 'E', 'EXHALL_000_100', 'EXHALL_001_101', 'EXHALL_010_110', 'EXHALL_011_111', 'EYHALL_000_010', 'EYHALL_001_011', 'EYHALL_100_110', 'EYHALL_101_111', 'EZHALL_000_001', 'EZHALL_010_011', 'EZHALL_100_101', 'EZHALL_110_111', 'E_vol', 'LB_weight', 'MPI_rank', 'PERB_vol', 'PTensorBackstreamDiagonal', 'PTensorBackstreamOffDiagonal', 'PTensorDiagonal', 'PTensorNonBackstreamDiagonal', 'PTensorNonBackstreamOffDiagonal', 'PTensorOffDiagonal', 'Pressure_f

...


=== 3d FIH ===
file: /wrk-vakka/group/spacephysics/vlasiator/3D/FIH/bulk5/bulk5.0000207.vlsv
active_populations: ['proton']
avgs check_population: False
proton check_population: True
protons check_population: False
vxblocks_ini check_parameter: False
vxblocks check_parameter: False
vxmin check_parameter: False
vxmax check_parameter: False
avgs velocity mesh size error: Error: variable vxblocks_ini/PARAMETER//pass not found in .vlsv file or in data reducers!
 Reader file /wrk-vakka/group/spacephysics/vlasiator/3D/FIH/bulk5/bulk5.0000207.vlsv
proton velocity mesh size: [70 70 70]
protons velocity mesh size error: Error: variable vxblocks_ini/PARAMETER//pass not found in .vlsv file or in data reducers!
 Reader file /wrk-vakka/group/spacephysics/vlasiator/3D/FIH/bulk5/bulk5.0000207.vlsv
first variables: ['CellID', 'ig_cellarea', 'ig_e', 'ig_electrontemp', 'ig_fac', 'ig_inplanecurrent', 'ig_latitude', 'ig_openclosed', 'ig_potential', 'ig_precipitation', 'ig_rhon', 'ig_sigmah', 'ig_sigmap',

INFO: Found population proton



=== 3d FII ===
file: /wrk-vakka/group/spacephysics/vlasiator/3D/FII/bulk5/bulk5.0000207.vlsv
active_populations: ['proton']
avgs check_population: False
proton check_population: True
protons check_population: False
vxblocks_ini check_parameter: False
vxblocks check_parameter: False
vxmin check_parameter: False
vxmax check_parameter: False
avgs velocity mesh size error: Error: variable vxblocks_ini/PARAMETER//pass not found in .vlsv file or in data reducers!
 Reader file /wrk-vakka/group/spacephysics/vlasiator/3D/FII/bulk5/bulk5.0000207.vlsv
proton velocity mesh size: [70 70 70]
protons velocity mesh size error: Error: variable vxblocks_ini/PARAMETER//pass not found in .vlsv file or in data reducers!
 Reader file /wrk-vakka/group/spacephysics/vlasiator/3D/FII/bulk5/bulk5.0000207.vlsv
first variables: ['CellID', 'ig_cellarea', 'ig_e', 'ig_electrontemp', 'ig_fac', 'ig_inplanecurrent', 'ig_latitude', 'ig_openclosed', 'ig_potential', 'ig_precipitation', 'ig_rhon', 'ig_sigmah', 'ig_sigmap',

INFO: Found population proton



=== 3d FJA ===
file: /wrk-vakka/group/spacephysics/vlasiator/3D/FJA/bulk1/bulk1.0001638.vlsv
active_populations: ['proton']
avgs check_population: False
proton check_population: True
protons check_population: False
vxblocks_ini check_parameter: False
vxblocks check_parameter: False
vxmin check_parameter: False
vxmax check_parameter: False
avgs velocity mesh size error: Error: variable vxblocks_ini/PARAMETER//pass not found in .vlsv file or in data reducers!
 Reader file /wrk-vakka/group/spacephysics/vlasiator/3D/FJA/bulk1/bulk1.0001638.vlsv
proton velocity mesh size: [70 70 70]
protons velocity mesh size error: Error: variable vxblocks_ini/PARAMETER//pass not found in .vlsv file or in data reducers!
 Reader file /wrk-vakka/group/spacephysics/vlasiator/3D/FJA/bulk1/bulk1.0001638.vlsv
first variables: ['CellID', 'ig_b', 'ig_cellarea', 'ig_e', 'ig_electrontemp', 'ig_fac', 'ig_inplanecurrent', 'ig_latitude', 'ig_openclosed', 'ig_potential', 'ig_precipitation', 'ig_rhon', 'ig_sigmah', 'ig_

In [ ]:
t = 2200
VDFlim = 2e6
R_EARTH = 6.371e6
cellid = 3601701

labels = np.array([1, 0])
label_names = ["x_point", "lobe"]

fileLocation = "/wrk-vakka/group/spacephysics/vlasiator/2D/BFD/bulk/"
bulkName = "bulk.000"+str(t)+".vlsv"
fileName = fileLocation + bulkName

file = pt.vlsvfile.VlsvReader(fileName)

file.list()

coords_m_1 = file.get_cell_coordinates(3601751)
print(coords_m_1)
coords_re_1 = np.asarray(coords_m_1) / R_EARTH
print(coords_re_1)
coords_m_2 = file.get_cell_coordinates(3601701)
print(coords_m_2)
coords_re_2 = np.asarray(coords_m_2) / R_EARTH
print(coords_re_2)
coords_m_3 = file.get_cell_coordinates(3601651)
print(coords_m_3)
coords_re_3= np.asarray(coords_m_3) / R_EARTH
print(coords_re_3)

coords_m_4 = file.get_cell_coordinates(3601801)
print(coords_m_4)
coords_re_4 = np.asarray(coords_m_4) / R_EARTH
print(coords_re_4)
coords_m_5 = file.get_cell_coordinates(3601851)
print(coords_m_5)
coords_re_5 = np.asarray(coords_m_5) / R_EARTH
print(coords_re_5)
coords_m_6 = file.get_cell_coordinates(3451751)
print(coords_m_6)
coords_re_6= np.asarray(coords_m_6) / R_EARTH
print(coords_re_6)

coords_m_7 = file.get_cell_coordinates(3751751)
print(coords_m_7)
coords_re_7= np.asarray(coords_m_7) / R_EARTH
print(coords_re_7)



coords = np.array([coords_re_1, coords_re_2, coords_re_3, coords_re_4, coords_re_5, coords_re_6, coords_re_7])

In [ ]:
def expr_velocity(exprmaps, requestvariables=False):
    """
    Return bilk velocity for Analysator colormap plotting

    Parameters
    ----------
    exprmaps : dict 
        Dictionary containing the variables requested from the `.vlsv` file.
    requestvariables : bool, optional
        If true, return the names of reqiuired variables.

    returns
    -------
    list[str] or numpy.ndarray
        Required variable names, or the bulk velocity vector field.
    """

    if requestvariables is True:
        return ['rho', 'rho_v']

    rho = exprmaps['rho'][:,:]
    rhov = exprmaps['rho_v'][:,:,:]

    velocity = rhov / rho[:,:,None]

    return velocity

def extract_vdf(file,cid,box=-1, pop="avgs"):
    """
    Extract a 3D VDF from one Vlasiator cell.

    This function reads the sparse velocity-space data stored in a `.vlsv`
    file, places the values into a full velocity-space grid, sorts the grid by
    velocity coordinates, and returns the VDF as a dence NumPy array.

    Parameters
    ----------
    file : str
        Path to the `.vlsv` file.
    cid : int
        Spatial cell ID from which the VDF is extracted.
    box : int, optional
        If positive, crop the VDF around its maximum value using this value as
        the half-width of the crop in index space. If `-1`, return VDF.
    pop : str, optional
        Particle population name used by Analysator.

    Returns
    -------
    numpy.ndarray
        Dense 3D VDF array with dtype `float32`. The returned axis order `[vx, vy, vz]`.
    """

    assert cid>0
    f = pt.vlsvfile.VlsvReader(file)
    #read phase space density
    vcells = f.read_velocity_cells(cid, pop)
    keys = list(vcells.keys())
    values = list(vcells.values())

    #generate a velocity space
    size = f.get_velocity_mesh_size(pop)
    vids = np.arange(4 * 4 * 4 * int(size[0]) * int(size[1]) * int(size[2]))

    #put phase space density into array
    dist = np.zeros_like(vids,dtype=float)
    dist[keys] = values

    #sort vspace by velocity
    v = f.get_velocity_cell_coordinates(vids, pop)

    i = np.argsort(v[:,0],kind='stable')
    v = v[i]
    #vids = vids[i]
    dist = dist[i]

    j = np.argsort(v[:,1],kind='stable')
    v = v[j]
    #vids = vids[j]
    dist = dist[j]

    k = np.argsort(v[:,2],kind='stable')
    v = v[k]
    #vids = vids[k]
    dist = dist[k]
    dist = dist.reshape(4*int(size[0]),4*int(size[1]),4*int(size[2]))
    vdf=dist
    i,j,k = np.unravel_index(np.argmax(vdf), vdf.shape)
    len=int(box)
    if box >0:
        data=vdf[(i-len):(i+len),(j-len):(j+len),(k-len):(k+len)]
    else:
        data =vdf

    data=np.swapaxes(data,2,0)
    return np.array(data,dtype=np.float32)

In [ ]:
fig = plt.figure(figsize=(15, 22))
ax1 = fig.add_subplot(411)

pt.plot.plot_colormap(
    filename=fileName,
    axes=ax1,
    boxre=[-40, 5, -6, 6],
    var = "V",
    #expression=expr_velocity,
    #operator="x",
    vmin=-1.5e6,
    vmax=1.5e6,
    streamlines="B",
    streamlinecolor="black",
)

ax1.scatter(
    -12.748156505248897,
    0.009525318118978205,
    marker="x",
    s=70,
    color="blue",
    label="x point"
)

ax1.scatter(
    coords[:, 0],
    coords[:, 2],
    marker=".",
    s=70,
    color="red",
    label="Neighbor VDF cells",
)

for i, coord in enumerate(coords, start=1):
    ax1.annotate(
        str(i),
        xy=(coord[0], coord[2]),
        xytext=(5, 5),
        textcoords="offset points",
        color="red",
        fontsize=10,
        weight="bold",
    )

ax1.legend()

In [ ]:
cids = []

for coord in coords:
    cid = file.get_cellid_with_vdf(coord, pop="ion")
    pt.plot.plot_vdf(filename=fileName,
                     cellids=[cid],
                     box=[-VDFlim,VDFlim,-VDFlim,VDFlim],
                     colormap='nipy_spectral',
                     xz=1,
                     draw=1)
    cids.append(cid)

# Experimentation ongoing

In [ ]:
extent = file.get_velocity_mesh_extent(pop="avgs")

X = []
y = []
metadata = []

for i, (cid, label) in enumerate(zip(cids, labels)):
    vdf = extract_vdf(
        fileName,
        cid=int(cid),
        )

    X.append(vdf)
    y.append(label)

    metadata.append({"sample_index": i,
                     "cid": int(cid),
                     "lable": int(label),
                     "class_name": "x_point" if int(label) == 1 else "lobe",})

X = np.stack(X).astype(np.float32)
y = np.array(y, dtype=np.int64)

In [ ]:
project_root = Path.home() / "vdf-ml"
outdir = project_root / "data" / "notebooks" / "01_vdf_inspecting" / f"timestep_{int(t)}"
outdir.mkdir(parents=True, exist_ok=True)

In [ ]:
np.savez_compressed(
    outdir / "vdf.npz",
    X=X,
    y=y,
    extent=extent
)

pd.DataFrame(metadata).to_csv(
    outdir / "metadata.csv",
    index=False
)

In [ ]:
project_root = Path.home() / "vdf-ml"
data_dir = project_root / "data" / "notebooks" / "01_vdf_inspecting" / f"timestep_{int(t)}"

data = np.load(data_dir / "vdf.npz")
metadata = pd.read_csv(data_dir / "metadata.csv")
X = data["X"]
y = data["y"]
extent = data["extent"]

print(data)
print(X.shape)
print(y.shape)
print(X)
print(y)
print(extent)
display(metadata)

In [ ]:
dv = 30000.0

vxmin = extent[0]
vymin = extent[1]
vzmin = extent[2]
vxmax = extent[3]
vymax = extent[4]
vzmax = extent[5]

for i, vdf in enumerate(X):
    
    vdf_swapped=np.swapaxes(vdf,2,0)

    fig, ax1 = plt.subplots(figsize=(7, 6))

    mid = vdf_swapped.shape[1] // 2

    vdf_plot = vdf_swapped[:, mid, :] * dv
    vdf_plot = np.where(vdf_plot < 8.301134972025815e-16 * dv, 0, vdf_plot)
    vdf_plot = np.ma.masked_less_equal(vdf_plot, 0)

    im = ax1.imshow(
        vdf_plot,
        origin="lower",
        extent=[
        vxmin / 1000, vxmax / 1000,
        vzmin / 1000, vzmax / 1000
    ],

        norm="log",
        cmap="nipy_spectral",
    )

    ax1.grid(color="gray", axis="both")
    ax1.set_xlim(-VDFlim / 1000, VDFlim / 1000)
    ax1.set_ylim(-VDFlim / 1000, VDFlim / 1000)
    ax1.set_xlabel("v_x")
    ax1.set_ylabel("v_z")

    ax1.set_title(f"sample {i}, label={y[i]}")

    fig.colorbar(im, ax=ax1, label=r"f(v)")
    plt.show()